# GARCH-in-Mean: Premio de Risco e Volatilidade

Neste notebook, exploraremos o modelo **GARCH-M** (GARCH-in-Mean),
proposto por **Engle, Lilien & Robins (1987)**.

O GARCH-M permite que a **volatilidade condicional** entre diretamente na
equacao da media, capturando o **premio de risco** — a compensacao adicional
que investidores exigem por assumir maior risco.

**Conteudo:**
1. Relacao risco-retorno
2. O modelo GARCH-M
3. Variantes do GARCH-M
4. Interpretacao do parametro $\lambda$
5. EGARCH-M
6. Exercicios

**Referencias:**
- Engle, R.F., Lilien, D.M. & Robins, R.P. (1987). *Estimating time varying risk premia in the term structure: The ARCH-M model*. Econometrica, 55(2), 391-407.
- Bollerslev, T., Engle, R.F. & Wooldridge, J.M. (1988). *A capital asset pricing model with time-varying covariances*. Journal of Political Economy, 96(1), 116-131.
- Nelson, D.B. (1991). *Conditional heteroskedasticity in asset returns: A new approach*. Econometrica, 59(2), 347-370.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from utils.plot_helpers import plot_garchm_risk_premium

from archbox.models import EGARCH, GARCH, GARCHM

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

## 1. Relacao risco-retorno

A teoria financeira classica (CAPM, ICAPM) preve que ativos mais arriscados devem
oferecer retornos esperados maiores. Formalmente:

$$E[r_t | \mathcal{F}_{t-1}] = \mu + \lambda \cdot \text{Risco}_t$$

onde $\lambda > 0$ e o **preco do risco** (risk premium per unit of risk).

O **ICAPM** de Merton (1973) sugere que o retorno esperado de um ativo depende
da sua **variancia condicional**:

$$E[r_t | \mathcal{F}_{t-1}] = \mu + \lambda \cdot \sigma_t^2$$

Vamos verificar empiricamente se existe relacao positiva entre volatilidade e retorno.

In [ ]:
# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

# Calcular volatilidade realizada rolling e retorno medio rolling
vol_20 = returns.rolling(20).std()
ret_20 = returns.rolling(20).mean()

# Remover NaNs
mask = vol_20.notna() & ret_20.notna()
vol_clean = vol_20[mask]
ret_clean = ret_20[mask]

# Scatter plot: retorno vs volatilidade realizada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Scatter com linha de regressao
axes[0].scatter(vol_clean, ret_clean, alpha=0.3, s=10, color='steelblue')
coeffs = np.polyfit(vol_clean, ret_clean, 1)
x_line = np.linspace(vol_clean.min(), vol_clean.max(), 100)
axes[0].plot(x_line, np.polyval(coeffs, x_line), 'r-', linewidth=2,
             label=f'y = {coeffs[0]:.4f}x + {coeffs[1]:.6f}')
axes[0].set_xlabel('Volatilidade Rolling (20 dias)')
axes[0].set_ylabel('Retorno Medio Rolling (20 dias)')
axes[0].set_title('Relacao Risco-Retorno Empirica')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Subplot 2: Series temporais sobrepostas
ax2 = axes[1]
ax2.plot(data.index, returns, alpha=0.3, linewidth=0.5, color='steelblue', label='Retornos')
ax2_twin = ax2.twinx()
ax2_twin.plot(data.index, vol_20, color='red', alpha=0.7, linewidth=0.8, label='Vol Rolling 20d')
ax2.set_xlabel('Data')
ax2.set_ylabel('Retornos', color='steelblue')
ax2_twin.set_ylabel('Volatilidade', color='red')
ax2.set_title('Retornos e Volatilidade ao Longo do Tempo')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlacao
corr = np.corrcoef(vol_clean, ret_clean)[0, 1]
print(f'Correlacao (vol, retorno): {corr:.4f}')
print(f'Coeficiente angular: {coeffs[0]:.6f}')
print('\nObservacao: a relacao pode ser fraca ou ate negativa em janelas curtas.')
print('Isso motiva o uso do GARCH-M para capturar a relacao de forma mais sofisticada.')

## 2. O modelo GARCH-M

O **GARCH-M** modela simultaneamente a media e a variancia condicional:

**Equacao da media:**
$$r_t = \mu + \lambda \cdot f(\sigma_t^2) + \epsilon_t$$

**Equacao da variancia:**
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

onde $\epsilon_t = \sigma_t z_t$ com $z_t \sim N(0,1)$.

O parametro $\lambda$ captura o **premio de risco**: se $\lambda > 0$,
periodos de alta volatilidade estao associados a maiores retornos esperados.

A funcao $f(\sigma_t^2)$ pode assumir diferentes formas:
- $f(\sigma_t^2) = \sigma_t$ (desvio padrao)
- $f(\sigma_t^2) = \sigma_t^2$ (variancia)
- $f(\sigma_t^2) = \log(\sigma_t^2)$ (log-variancia)

In [ ]:
# Estimar GARCH-M com f(sigma) = sigma (desvio padrao)
model_vol = GARCHM(returns.values, p=1, q=1, risk_premium='volatility')
results_vol = model_vol.fit()

# Exibir resultados
print(results_vol.summary())

# Extrair parametros
params = results_vol.params
param_names = results_vol.param_names

print('\n--- Parametros Estimados ---')
for name, val in zip(param_names, params, strict=False):
    print(f'{name:>10}: {val:.6f}')

# Lambda
lambda_hat = params[-1]
lambda_tstat = results_vol.tvalues[-1]
print(f'\nLambda (premio de risco): {lambda_hat:.6f}')
print(f't-statistic: {lambda_tstat:.4f}')
print(f'Interpretacao: cada unidade de aumento em sigma aumenta o retorno esperado em {lambda_hat:.6f}')

## 3. Variantes do GARCH-M

As tres formas mais comuns para $f(\sigma_t^2)$ sao:

| Variante | $f(\sigma_t^2)$ | Parametro archbox | Interpretacao de $\lambda$ |
|:---:|:---:|:---:|:---:|
| GARCH-M (vol) | $\sigma_t$ | `risk_premium='volatility'` | Premio por unidade de desvio padrao |
| GARCH-M (var) | $\sigma_t^2$ | `risk_premium='variance'` | Premio por unidade de variancia |
| GARCH-M (log) | $\log(\sigma_t^2)$ | `risk_premium='log_variance'` | Semi-elasticidade |

A escolha da forma funcional afeta a **interpretacao** do premio de risco,
mas nao a **estrutura** do modelo. Na pratica, a forma com $\sigma_t$
e mais intuitiva pois $\lambda$ tem a mesma unidade que o retorno.

In [ ]:
# Estimar as 3 variantes do GARCH-M

# Variante 1: volatility (sigma)
# Ja estimada acima: results_vol

# Variante 2: variance (sigma^2)
model_var = GARCHM(returns.values, p=1, q=1, risk_premium='variance')
results_var = model_var.fit()

# Variante 3: log_variance (log sigma^2)
model_log = GARCHM(returns.values, p=1, q=1, risk_premium='log_variance')
results_log = model_log.fit()

# Tabela comparativa
comparison = pd.DataFrame({
    'Variante': ['volatility (sigma)', 'variance (sigma^2)', 'log_variance (log sigma^2)'],
    'Lambda': [results_vol.params[-1], results_var.params[-1], results_log.params[-1]],
    'Lambda t-stat': [results_vol.tvalues[-1], results_var.tvalues[-1], results_log.tvalues[-1]],
    'Lambda p-value': [results_vol.pvalues[-1], results_var.pvalues[-1], results_log.pvalues[-1]],
    'omega': [results_vol.params[0], results_var.params[0], results_log.params[0]],
    'alpha': [results_vol.params[1], results_var.params[1], results_log.params[1]],
    'beta': [results_vol.params[2], results_var.params[2], results_log.params[2]],
    'Persistencia': [
        results_vol.params[1] + results_vol.params[2],
        results_var.params[1] + results_var.params[2],
        results_log.params[1] + results_log.params[2],
    ],
    'AIC': [results_vol.aic, results_var.aic, results_log.aic],
    'BIC': [results_vol.bic, results_var.bic, results_log.bic],
    'Log-Lik': [results_vol.loglike, results_var.loglike, results_log.loglike],
})

print('=== Comparacao das 3 Variantes do GARCH-M ===')
print(comparison.to_string(index=False, float_format='%.6f'))

print(f'\nMelhor variante por AIC: {comparison.loc[comparison["AIC"].idxmin(), "Variante"]}')
print(f'Melhor variante por BIC: {comparison.loc[comparison["BIC"].idxmin(), "Variante"]}')

## 4. Interpretacao do parametro $\lambda$

O parametro $\lambda$ e central no GARCH-M:

- **$\lambda > 0$**: premio de risco positivo — investidores exigem maior retorno
  esperado em periodos de alta volatilidade (consistente com teoria)
- **$\lambda = 0$**: sem premio de risco — volatilidade nao afeta o retorno esperado
  (o GARCH-M reduz ao GARCH padrao)
- **$\lambda < 0$**: premio de risco negativo — retornos menores em periodos de alta
  volatilidade (pode ocorrer em periodos de panico)

Para testar se $\lambda$ e estatisticamente significativo:

$$H_0: \lambda = 0 \quad \text{vs} \quad H_1: \lambda \neq 0$$

Usamos a estatistica $t = \hat{\lambda} / \text{se}(\hat{\lambda})$.

In [ ]:
# Teste de significancia de lambda (usando a variante volatility)
lambda_hat = results_vol.params[-1]
lambda_se = results_vol.std_errors[-1]
lambda_tstat = results_vol.tvalues[-1]
lambda_pval = results_vol.pvalues[-1]

print('=== Teste de Significancia do Premio de Risco ===')
print(f'Lambda estimado:  {lambda_hat:.6f}')
print(f'Erro padrao:      {lambda_se:.6f}')
print(f't-statistic:      {lambda_tstat:.4f}')
print(f'p-value:          {lambda_pval:.6f}')
print()
if lambda_pval < 0.05:
    print('Resultado: Rejeitamos H0 ao nivel 5%. Lambda e estatisticamente significativo.')
else:
    print('Resultado: Nao rejeitamos H0 ao nivel 5%. Lambda NAO e significativo.')
if lambda_hat > 0:
    print('Sinal positivo: consistente com a teoria de premio de risco.')
else:
    print('Sinal negativo: inconsistente com a teoria classica.')

# Visualizar o premio de risco ao longo do tempo
risk_premium = lambda_hat * results_vol.conditional_volatility

fig = plot_garchm_risk_premium(
    dates=data.index,
    returns=returns.values,
    conditional_vol=results_vol.conditional_volatility,
    risk_premium=risk_premium,
    title='GARCH-M: Premio de Risco Variante no Tempo'
)
plt.show()

# Estatisticas do premio de risco
print('\n--- Estatisticas do Premio de Risco ---')
print(f'Media:   {np.mean(risk_premium):.6f}')
print(f'Std:     {np.std(risk_premium):.6f}')
print(f'Min:     {np.min(risk_premium):.6f}')
print(f'Max:     {np.max(risk_premium):.6f}')
print(f'Mediana: {np.median(risk_premium):.6f}')

## 5. EGARCH-M

Podemos combinar o efeito de **assimetria** (leverage) com o **premio de risco**
usando um modelo EGARCH na equacao da variancia:

**Equacao da media:**
$$r_t = \mu + \lambda \cdot \sigma_t + \epsilon_t$$

**Equacao da variancia (EGARCH):**
$$\log(\sigma_t^2) = \omega + \alpha |z_{t-1}| + \gamma z_{t-1} + \beta \log(\sigma_{t-1}^2)$$

O parametro $\gamma < 0$ captura o **efeito alavancagem**: choques negativos
($z_{t-1} < 0$) aumentam mais a volatilidade do que choques positivos.

O EGARCH-M combina dois fatos estilizados:
1. Volatilidade assimetrica (efeito alavancagem)
2. Premio de risco variante no tempo

In [ ]:
# Estimar EGARCH puro para referencia
model_egarch = EGARCH(returns.values, p=1, q=1)
results_egarch = model_egarch.fit()

print('=== EGARCH(1,1) ===')
print(results_egarch.summary())

# Verificar efeito alavancagem (gamma)
egarch_params = results_egarch.params
egarch_names = results_egarch.param_names
print('\n--- Parametros EGARCH ---')
for name, val in zip(egarch_names, egarch_params, strict=False):
    print(f'{name:>10}: {val:.6f}')

# Gamma negativo => efeito alavancagem
gamma_idx = [i for i, n in enumerate(egarch_names) if 'gamma' in n.lower()]
if gamma_idx:
    gamma_hat = egarch_params[gamma_idx[0]]
    gamma_tstat = results_egarch.tvalues[gamma_idx[0]]
    gamma_pval = results_egarch.pvalues[gamma_idx[0]]
    print(f'\nEfeito alavancagem (gamma): {gamma_hat:.6f}')
    print(f't-stat: {gamma_tstat:.4f}, p-value: {gamma_pval:.6f}')
    if gamma_hat < 0:
        print('Gamma negativo: choques negativos aumentam MAIS a volatilidade (leverage effect).')

# Construir premio de risco usando EGARCH
# Usamos o lambda estimado do GARCH-M com a vol condicional do EGARCH
egarch_premium = lambda_hat * results_egarch.conditional_volatility

# Plot comparativo: volatilidade GARCH-M vs EGARCH
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(data.index, results_vol.conditional_volatility, alpha=0.7,
             linewidth=0.8, label='GARCH-M (volatility)', color='steelblue')
axes[0].plot(data.index, results_egarch.conditional_volatility, alpha=0.7,
             linewidth=0.8, label='EGARCH', color='red')
axes[0].set_ylabel('Volatilidade Condicional')
axes[0].set_title('Volatilidade Condicional: GARCH-M vs EGARCH')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(data.index, risk_premium, alpha=0.7,
             linewidth=0.8, label='Premio (GARCH-M)', color='steelblue')
axes[1].plot(data.index, egarch_premium, alpha=0.7,
             linewidth=0.8, label='Premio (EGARCH-based)', color='red')
axes[1].set_ylabel('Premio de Risco')
axes[1].set_xlabel('Data')
axes[1].set_title('Premio de Risco: GARCH-M vs EGARCH-based')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nAIC EGARCH: {results_egarch.aic:.4f}')
print(f'AIC GARCH-M: {results_vol.aic:.4f}')

## 6. Exercicios

1. **GARCH vs GARCH-M**: Estime um GARCH(1,1) padrao e compare com o GARCH-M.
   O premio de risco melhora o ajuste do modelo (menor AIC)?

2. **Distribuicao t-Student**: Re-estime o GARCH-M com distribuicao t-Student
   para os erros. O lambda muda significativamente?

3. **Subamostras**: Divida os dados em duas metades e estime o GARCH-M em cada
   uma. O lambda e estavel ao longo do tempo?

4. **Impacto economico**: Calcule a diferenca no retorno esperado entre um
   periodo de baixa volatilidade (percentil 10) e alta volatilidade (percentil 90).
   O premio de risco e economicamente significativo?

In [ ]:
# Exercicio: Compare GARCH(1,1) vs GARCH-M vs EGARCH

# Estimar GARCH(1,1) padrao
model_g = GARCH(returns.values, p=1, q=1)
res_g = model_g.fit()

# Tabela comparativa final
final_comparison = pd.DataFrame({
    'Modelo': ['GARCH(1,1)', 'GARCH-M (vol)', 'GARCH-M (var)', 'GARCH-M (log)', 'EGARCH(1,1)'],
    'N. Params': [
        len(res_g.params),
        len(results_vol.params),
        len(results_var.params),
        len(results_log.params),
        len(results_egarch.params),
    ],
    'Log-Lik': [
        res_g.loglike,
        results_vol.loglike,
        results_var.loglike,
        results_log.loglike,
        results_egarch.loglike,
    ],
    'AIC': [
        res_g.aic,
        results_vol.aic,
        results_var.aic,
        results_log.aic,
        results_egarch.aic,
    ],
    'BIC': [
        res_g.bic,
        results_vol.bic,
        results_var.bic,
        results_log.bic,
        results_egarch.bic,
    ],
})

print('=== Comparacao Final: GARCH vs GARCH-M vs EGARCH ===')
print(final_comparison.to_string(index=False, float_format='%.4f'))

best_aic = final_comparison.loc[final_comparison['AIC'].idxmin(), 'Modelo']
best_bic = final_comparison.loc[final_comparison['BIC'].idxmin(), 'Modelo']
print(f'\nMelhor modelo por AIC: {best_aic}')
print(f'Melhor modelo por BIC: {best_bic}')

# Plot comparativo das volatilidades condicionais
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(data.index, res_g.conditional_volatility, alpha=0.6, linewidth=0.8,
        label='GARCH(1,1)', color='steelblue')
ax.plot(data.index, results_vol.conditional_volatility, alpha=0.6, linewidth=0.8,
        label='GARCH-M (vol)', color='darkorange')
ax.plot(data.index, results_egarch.conditional_volatility, alpha=0.6, linewidth=0.8,
        label='EGARCH(1,1)', color='red')
ax.set_xlabel('Data')
ax.set_ylabel('Volatilidade Condicional')
ax.set_title('Volatilidade Condicional: GARCH vs GARCH-M vs EGARCH')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Impacto economico do premio de risco
cond_vol = results_vol.conditional_volatility
p10 = np.percentile(cond_vol, 10)
p90 = np.percentile(cond_vol, 90)
diff_premium = lambda_hat * (p90 - p10)

print('\n=== Impacto Economico do Premio de Risco ===')
print(f'Volatilidade percentil 10: {p10:.6f}')
print(f'Volatilidade percentil 90: {p90:.6f}')
print(f'Diferenca no retorno esperado: {diff_premium:.6f}')
print(f'Diferenca anualizada: {diff_premium * 252:.4f} ({diff_premium * 252 * 100:.2f}%)')